# Line Graph Transformation

This notebook demonstrates the line graph transformation using topologic_fast.

## What is a Line Graph?

The **line graph** L(G) of a graph G is a graph where:
- Each vertex in L(G) represents an edge in G
- Two vertices in L(G) are connected if their corresponding edges in G share a common vertex

In architectural terms:
- If G represents rooms (vertices) connected by doors (edges)
- Then L(G) represents doors (vertices) connected when they lead to the same room (edges)

**Note**: This notebook uses topologic_fast's native `Graph.LineGraph()` implementation.

In [ ]:
import topologic_fast as tf
import plotly.graph_objects as go
from plotly.subplots import make_subplots

## 1. Create a Sample Graph

We'll create a simple graph representing a building layout.

In [ ]:
# Create a sample graph: 5 rooms connected in a pattern
#
#     v1 --- v2
#     |  \    |
#     |   \   |
#     v4---v0--v3
#
# v0 is the central hub (lobby)

v0 = tf.Vertex.ByCoordinates(0, 0, 0)   # Central hub
v1 = tf.Vertex.ByCoordinates(-2, 2, 0)  # Top-left
v2 = tf.Vertex.ByCoordinates(2, 2, 0)   # Top-right
v3 = tf.Vertex.ByCoordinates(2, 0, 0)   # Right
v4 = tf.Vertex.ByCoordinates(-2, 0, 0)  # Left

# Create edges (connections between rooms)
e01 = tf.Edge.ByStartVertexEndVertex(v0, v1)  # Hub to top-left
e02 = tf.Edge.ByStartVertexEndVertex(v0, v2)  # Hub to top-right (diagonal)
e03 = tf.Edge.ByStartVertexEndVertex(v0, v3)  # Hub to right
e04 = tf.Edge.ByStartVertexEndVertex(v0, v4)  # Hub to left
e12 = tf.Edge.ByStartVertexEndVertex(v1, v2)  # Top-left to top-right
e14 = tf.Edge.ByStartVertexEndVertex(v1, v4)  # Top-left to left
e23 = tf.Edge.ByStartVertexEndVertex(v2, v3)  # Top-right to right

vertices = [v0, v1, v2, v3, v4]
edges = [e01, e02, e03, e04, e12, e14, e23]

original_graph = tf.Graph.ByVerticesEdges(vertices, edges)

print(f"Original Graph G:")
print(f"  Vertices (rooms): {original_graph.Order()}")
print(f"  Edges (doors):    {original_graph.Size()}")
print(f"  Density:          {original_graph.Density():.3f}")

## 2. Visualize the Original Graph

In [ ]:
def visualize_graph(graph, title="Graph", vertex_labels=None, edge_labels=None, 
                   vertex_color='blue', edge_color='gray'):
    """Create a visualization of a graph."""
    fig = go.Figure()
    
    graph_vertices = graph.Vertices()
    graph_edges = graph.Edges()
    
    # Draw edges
    for i, edge in enumerate(graph_edges):
        verts = edge.Vertices()
        if len(verts) == 2:
            p1 = verts[0].Coordinates()
            p2 = verts[1].Coordinates()
            
            # Draw edge line
            fig.add_trace(go.Scatter(
                x=[p1[0], p2[0]], y=[p1[1], p2[1]],
                mode='lines',
                line=dict(color=edge_color, width=3),
                showlegend=False,
                hoverinfo='skip'
            ))
            
            # Add edge label if provided
            if edge_labels:
                mid_x = (p1[0] + p2[0]) / 2
                mid_y = (p1[1] + p2[1]) / 2
                fig.add_annotation(
                    x=mid_x, y=mid_y,
                    text=edge_labels[i] if i < len(edge_labels) else f"e{i}",
                    showarrow=False,
                    font=dict(size=10, color='red'),
                    bgcolor='white'
                )
    
    # Draw vertices
    x_coords = [v.X() for v in graph_vertices]
    y_coords = [v.Y() for v in graph_vertices]
    
    hover_text = [vertex_labels[i] if vertex_labels and i < len(vertex_labels) else f"v{i}" 
                  for i in range(len(graph_vertices))]
    
    fig.add_trace(go.Scatter(
        x=x_coords, y=y_coords,
        mode='markers+text',
        marker=dict(size=30, color=vertex_color, line=dict(color='black', width=2)),
        text=hover_text,
        textposition='middle center',
        textfont=dict(size=12, color='white'),
        hovertext=hover_text,
        hoverinfo='text',
        showlegend=False
    ))
    
    fig.update_layout(
        title=title,
        xaxis=dict(scaleanchor='y', scaleratio=1, showgrid=True, zeroline=True),
        yaxis=dict(showgrid=True, zeroline=True),
        width=600,
        height=500
    )
    
    return fig


# Visualize original graph
vertex_labels = ['v0', 'v1', 'v2', 'v3', 'v4']
edge_labels = ['e01', 'e02', 'e03', 'e04', 'e12', 'e14', 'e23']

fig_original = visualize_graph(
    original_graph, 
    title="Original Graph G (5 vertices, 7 edges)",
    vertex_labels=vertex_labels,
    edge_labels=edge_labels
)
fig_original.show()

# Compute the line graph using topologic_fast's native implementation
line_graph = original_graph.LineGraph()

print(f"Line Graph L(G):")
print(f"  Vertices (original edges): {line_graph.Order()}")
print(f"  Edges (shared vertices):   {line_graph.Size()}")
print(f"  Density:                   {line_graph.Density():.3f}")

## 4. Visualize the Line Graph

## 4. Visualize the Line Graph

In [ ]:
# Create labels for line graph vertices (named after original edges)
line_vertex_labels = ['e01', 'e02', 'e03', 'e04', 'e12', 'e14', 'e23']

fig_line = visualize_graph(
    line_graph,
    title=f"Line Graph L(G) ({line_graph.Order()} vertices, {line_graph.Size()} edges)",
    vertex_labels=line_vertex_labels,
    vertex_color='green',
    edge_color='darkgreen'
)
fig_line.show()

## 5. Compare Original and Line Graph Side by Side

In [ ]:
fig_compare = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        f'Original Graph G ({original_graph.Order()} V, {original_graph.Size()} E)',
        f'Line Graph L(G) ({line_graph.Order()} V, {line_graph.Size()} E)'
    ]
)

# Draw original graph
orig_verts = original_graph.Vertices()
orig_edges = original_graph.Edges()

for i, edge in enumerate(orig_edges):
    verts = edge.Vertices()
    if len(verts) == 2:
        p1 = verts[0].Coordinates()
        p2 = verts[1].Coordinates()
        fig_compare.add_trace(go.Scatter(
            x=[p1[0], p2[0]], y=[p1[1], p2[1]],
            mode='lines',
            line=dict(color='gray', width=3),
            showlegend=False
        ), row=1, col=1)
        
        # Add edge label
        mid_x = (p1[0] + p2[0]) / 2
        mid_y = (p1[1] + p2[1]) / 2
        fig_compare.add_annotation(
            x=mid_x, y=mid_y,
            text=edge_labels[i],
            showarrow=False,
            font=dict(size=9, color='red'),
            bgcolor='white',
            xref='x', yref='y'
        )

fig_compare.add_trace(go.Scatter(
    x=[v.X() for v in orig_verts],
    y=[v.Y() for v in orig_verts],
    mode='markers+text',
    marker=dict(size=25, color='blue', line=dict(color='black', width=2)),
    text=vertex_labels,
    textposition='middle center',
    textfont=dict(size=10, color='white'),
    showlegend=False
), row=1, col=1)

# Draw line graph
line_verts = line_graph.Vertices()
line_edges = line_graph.Edges()

for edge in line_edges:
    verts = edge.Vertices()
    if len(verts) == 2:
        p1 = verts[0].Coordinates()
        p2 = verts[1].Coordinates()
        fig_compare.add_trace(go.Scatter(
            x=[p1[0], p2[0]], y=[p1[1], p2[1]],
            mode='lines',
            line=dict(color='darkgreen', width=2),
            showlegend=False
        ), row=1, col=2)

fig_compare.add_trace(go.Scatter(
    x=[v.X() for v in line_verts],
    y=[v.Y() for v in line_verts],
    mode='markers+text',
    marker=dict(size=25, color='green', line=dict(color='black', width=2)),
    text=line_vertex_labels,
    textposition='middle center',
    textfont=dict(size=8, color='white'),
    showlegend=False
), row=1, col=2)

fig_compare.update_xaxes(scaleanchor="y", scaleratio=1)
fig_compare.update_layout(
    title='Original Graph vs Line Graph Transformation',
    height=500,
    width=1100
)

fig_compare.show()

## 6. Line Graph Properties and Analysis

In [ ]:
# Analyze vertex degrees in line graph
print("Line Graph Vertex Degrees:")
print("(Each vertex represents an original edge, degree = number of adjacent edges)")
print("-" * 60)

line_verts = line_graph.Vertices()
for i, (v, label) in enumerate(zip(line_verts, line_vertex_labels)):
    degree = line_graph.VertexDegree(v)
    print(f"  {label}: degree {degree}")

print(f"\nDegree Sequence: {line_graph.DegreeSequence()}")
print(f"Max Degree: {line_graph.MaximumDelta()}")
print(f"Min Degree: {line_graph.MinimumDelta()}")

In [ ]:
# Graph comparison table
print("\n" + "=" * 50)
print("Graph Comparison")
print("=" * 50)
print(f"{'Metric':<25} {'G':<12} {'L(G)':<12}")
print("-" * 50)
print(f"{'Vertices (Order)':<25} {original_graph.Order():<12} {line_graph.Order():<12}")
print(f"{'Edges (Size)':<25} {original_graph.Size():<12} {line_graph.Size():<12}")
print(f"{'Density':<25} {original_graph.Density():<12.3f} {line_graph.Density():<12.3f}")
print(f"{'Diameter':<25} {original_graph.Diameter():<12} {line_graph.Diameter():<12}")
print(f"{'Max Degree':<25} {original_graph.MaximumDelta():<12} {line_graph.MaximumDelta():<12}")
print(f"{'Min Degree':<25} {original_graph.MinimumDelta():<12} {line_graph.MinimumDelta():<12}")
print(f"{'Is Bipartite':<25} {str(original_graph.IsBipartite()):<12} {str(line_graph.IsBipartite()):<12}")
print(f"{'Is Complete':<25} {str(original_graph.IsComplete()):<12} {str(line_graph.IsComplete()):<12}")

## 7. Architectural Example: Building Floor Plan

Create a line graph from a building layout to analyze door connectivity.

In [ ]:
# Create a simple building layout
rooms = [
    tf.Cell.Box(0, 0, 0, 3, 3, 3),    # Lobby
    tf.Cell.Box(3, 0, 0, 2, 3, 3),    # Office 1
    tf.Cell.Box(5, 0, 0, 2, 3, 3),    # Office 2
    tf.Cell.Box(0, 3, 0, 3, 2, 3),    # Corridor
    tf.Cell.Box(3, 3, 0, 4, 2, 3),    # Conference Room
]

room_names = ['Lobby', 'Office1', 'Office2', 'Corridor', 'Conference']

# Create CellComplex
building = tf.CellComplex.ByCells(rooms)

# Create dual graph (rooms as vertices, shared walls as edges)
room_graph = tf.Graph.ByTopology(building)

print(f"Building Layout Graph:")
print(f"  Rooms:       {room_graph.Order()}")
print(f"  Connections: {room_graph.Size()}")

In [ ]:
# Compute line graph of the building using native implementation
building_line_graph = room_graph.LineGraph()

print(f"Building Line Graph:")
print(f"  Vertices (doors):  {building_line_graph.Order()}")
print(f"  Connections:       {building_line_graph.Size()}")
print(f"")
print("Interpretation:")
print("  - Each vertex in L(G) represents a door/connection between rooms")
print("  - Edges in L(G) connect doors that lead to the same room")

In [ ]:
# Visualize building graphs
fig_building = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        'Room Graph G (rooms = vertices)',
        'Line Graph L(G) (doors = vertices)'
    ]
)

# Draw room graph
room_verts = room_graph.Vertices()
room_edges = room_graph.Edges()

for edge in room_edges:
    verts = edge.Vertices()
    if len(verts) == 2:
        p1 = verts[0].Coordinates()
        p2 = verts[1].Coordinates()
        fig_building.add_trace(go.Scatter(
            x=[p1[0], p2[0]], y=[p1[1], p2[1]],
            mode='lines',
            line=dict(color='gray', width=3),
            showlegend=False
        ), row=1, col=1)

fig_building.add_trace(go.Scatter(
    x=[v.X() for v in room_verts],
    y=[v.Y() for v in room_verts],
    mode='markers+text',
    marker=dict(size=30, color='blue', line=dict(color='black', width=2)),
    text=[n[:4] for n in room_names],
    textposition='middle center',
    textfont=dict(size=8, color='white'),
    hovertext=room_names,
    hoverinfo='text',
    showlegend=False
), row=1, col=1)

# Draw line graph
line_verts = building_line_graph.Vertices()
line_edges = building_line_graph.Edges()

for edge in line_edges:
    verts = edge.Vertices()
    if len(verts) == 2:
        p1 = verts[0].Coordinates()
        p2 = verts[1].Coordinates()
        fig_building.add_trace(go.Scatter(
            x=[p1[0], p2[0]], y=[p1[1], p2[1]],
            mode='lines',
            line=dict(color='darkgreen', width=2),
            showlegend=False
        ), row=1, col=2)

fig_building.add_trace(go.Scatter(
    x=[v.X() for v in line_verts],
    y=[v.Y() for v in line_verts],
    mode='markers',
    marker=dict(size=20, color='green', line=dict(color='black', width=2)),
    name='Doors',
    showlegend=False
), row=1, col=2)

fig_building.update_xaxes(scaleanchor="y", scaleratio=1)
fig_building.update_layout(
    title='Building: Room Graph vs Door Line Graph',
    height=500,
    width=1000
)

fig_building.show()

## 8. Iterated Line Graphs

We can compute L(L(G)), L(L(L(G))), etc. The number of vertices and edges typically grows.

In [ ]:
# Compute iterated line graphs using native implementation
current_graph = original_graph
iterations = []

print("Iterated Line Graph Sequence:")
print("=" * 50)
print(f"{'Iteration':<12} {'Vertices':<12} {'Edges':<12} {'Density':<12}")
print("-" * 50)

for i in range(5):
    v = current_graph.Order()
    e = current_graph.Size()
    d = current_graph.Density()
    iterations.append((i, v, e, d))
    
    name = "G" if i == 0 else f"L^{i}(G)"
    print(f"{name:<12} {v:<12} {e:<12} {d:<12.3f}")
    
    if e == 0:
        print("\nStopped: No edges to transform.")
        break
    
    current_graph = current_graph.LineGraph()

In [ ]:
# Plot the growth
fig_growth = go.Figure()

iterations_data = [(i, v, e, d) for i, v, e, d in iterations]

fig_growth.add_trace(go.Scatter(
    x=[i[0] for i in iterations_data],
    y=[i[1] for i in iterations_data],
    mode='lines+markers',
    name='Vertices',
    marker=dict(size=10)
))

fig_growth.add_trace(go.Scatter(
    x=[i[0] for i in iterations_data],
    y=[i[2] for i in iterations_data],
    mode='lines+markers',
    name='Edges',
    marker=dict(size=10)
))

fig_growth.update_layout(
    title='Growth of Iterated Line Graphs',
    xaxis=dict(title='Iteration'),
    yaxis=dict(title='Count', type='log'),
    width=700,
    height=400
)

fig_growth.show()

## Summary

This notebook demonstrated the line graph transformation:

### Key Concepts

1. **Line Graph Definition**: L(G) has:
   - Vertices corresponding to edges in G
   - Edges connecting vertices whose original edges shared a vertex

2. **Properties**:
   - |V(L(G))| = |E(G)|
   - Two vertices in L(G) are adjacent iff their corresponding edges in G are incident
   - The line graph is always claw-free

3. **Applications**:
   - Analyzing edge relationships in networks
   - Studying door/connection patterns in buildings
   - Understanding how edges cluster around vertices

### topologic_fast Methods Used

- `tf.Graph.LineGraph()` - Compute the line graph transformation
- `tf.Vertex.ByCoordinates()` - Create vertices
- `tf.Edge.ByStartVertexEndVertex()` - Create edges
- `tf.Graph.ByVerticesEdges()` - Create graphs
- `tf.Graph.ByTopology()` - Create dual graph from topology
- `graph.Order()`, `graph.Size()`, etc. - Graph metrics
- `tf.Cell.Box()`, `tf.CellComplex.ByCells()` - Create building geometry